<a href="https://colab.research.google.com/github/jxy-ygjm/EC1B1-Coursework/blob/main/EC1B1_Code_ipynb_Group_55.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EC1B1 Coursework

**Winter Term 2025/2026**

<div style="font-family: system-ui; padding: 20px 30px 20px 20px; background-color: #FFFFFF; border-left: 8px solid #0570b0; border-radius: 8px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1);max-width:600px;color:#212121;">

**Coursework Notebook**
- Group 55
    - Xuanyang Ji
    - Felix Kalu
    - Rishi Ganiger
- Date: 28/02/2026

</div>

**Importing libraries**

To run the notebook, we first need to import all the libraries below

In [1]:
import numpy as np
import pandas as pd
from datetime import datetime

## Section 1: Importing the Data

In 24/03/2025, International Financial Statistics (IFS) data were discontinued as a single dataset. IMF deleted the "Query" function, so in this Notebook, we download data from different datasets separately.

The raw data are stored on [GitHub](https://github.com/jxy-ygjm/EC1B1-Coursework), allowing easier data import.

We store all the raw data in a dictionary, so we can easily handle procedures like merge

In [2]:
# Raw URL of data
raw_url = (
    "https://raw.githubusercontent.com/jxy-ygjm/EC1B1-Coursework/main/data/raw/"
)

# Specific data name
data_name = [
    "consumer_price", "exchange_rate", "industrial_production",
    "international_reserves"
]

In [3]:
# Create empty dictionary storing data
data = {}

# Read data
for name in data_name:
    data[name] = pd.read_csv(raw_url + name + ".csv")

Now, we try to investigate the dataframes

In [4]:
for value in data.values():
    print(value.columns)

Index(['COUNTRY', 'INDEX_TYPE', 'COICOP_1999', 'TYPE_OF_TRANSFORMATION',
       'FREQUENCY', 'TIME_PERIOD', 'OBS_VALUE', 'SCALE'],
      dtype='object')
Index(['COUNTRY', 'INDICATOR', 'TYPE_OF_TRANSFORMATION', 'FREQUENCY',
       'TIME_PERIOD', 'OBS_VALUE', 'SCALE'],
      dtype='object')
Index(['COUNTRY', 'PRODUCTION_INDEX', 'TYPE_OF_TRANSFORMATION', 'FREQUENCY',
       'TIME_PERIOD', 'OBS_VALUE', 'SCALE'],
      dtype='object')
Index(['COUNTRY', 'INDICATOR', 'UNIT', 'FREQUENCY', 'TIME_PERIOD', 'OBS_VALUE',
       'SCALE'],
      dtype='object')


We can see that only the `TIME_PERIOD` and `OBS_VALUE` are useful to us.

We now try to create a single dataframe for Spain and US

In [5]:
# Set a list for clean Spain data
spain_cleaned = []

for name, df in data.items():
    temp = df[df["COUNTRY"] == "Spain"][["TIME_PERIOD", "OBS_VALUE"]].copy()
    temp = temp.rename(columns={"OBS_VALUE": name})
    temp = temp.set_index("TIME_PERIOD")
    spain_cleaned.append(temp)

df_spain = pd.concat(spain_cleaned, axis=1)

In [6]:
# Investigate the dataframe
df_spain.head()

,consumer_price,exchange_rate,industrial_production,international_reserves
TIME_PERIOD,,,,
1959-M12,2.525752,60.0,NaN,200.14101
1960-M01,2.505928,60.0,NaN,233.26019
1960-M02,2.503895,60.0,NaN,253.21561
1960-M03,2.500846,60.0,NaN,299.28248
1960-M04,2.502879,60.0,NaN,326.28248


In [7]:
# Set a list for clean US data
us_cleaned = []

for name, df in data.items():
    temp = (
        df[df["COUNTRY"] == "United States"]
        [["TIME_PERIOD", "OBS_VALUE"]]
        .copy()
    )
    temp = temp.rename(columns={"OBS_VALUE": name})
    temp = temp.set_index("TIME_PERIOD")
    us_cleaned.append(temp)

df_us = pd.concat(us_cleaned, axis=1)

In [8]:
# Investigate the dataframe
df_us.head()

,consumer_price,exchange_rate,industrial_production,international_reserves
TIME_PERIOD,,,,
1959-M12,13.482806,NaN,NaN,21543.4138
1960-M01,13.436946,NaN,NaN,21539.3167
1960-M02,13.482806,NaN,NaN,21445.6179
1960-M03,13.482806,NaN,NaN,21411.2592
1960-M04,13.528666,NaN,NaN,21344.4744


Remember no data for `exchange_rate` and `industrial_production` so drop them

In [9]:
df_us = df_us.drop(columns=["exchange_rate", "industrial_production"])

As `consumer_price`, `exchange_rate` and `industrial_production` are indices or have direct meaning, we do not need to know their scale, but for international_reserves, as its nominal measure, we need to know its scale

In [10]:
data["international_reserves"]["SCALE"].head()

,SCALE
0,Millions
1,Millions
2,Millions
3,Millions
4,Millions


Can see all the data are in Millions scale, we can rename the column to highlight this

In [11]:
df_spain = df_spain.rename(
    columns={"international_reserves": "international_reserves (M)"}
)
df_us = df_us.rename(
    columns={"international_reserves": "international_reserves (M)"}
)

In [12]:
df_spain.head()

,consumer_price,exchange_rate,industrial_production,international_reserves (M)
TIME_PERIOD,,,,
1959-M12,2.525752,60.0,NaN,200.14101
1960-M01,2.505928,60.0,NaN,233.26019
1960-M02,2.503895,60.0,NaN,253.21561
1960-M03,2.500846,60.0,NaN,299.28248
1960-M04,2.502879,60.0,NaN,326.28248


## Section 2: Cleaning the Data

### The monthly growth in the nominal exchange rate

In [13]:
# Set the name of the country
df_spain["COUNTRY"] = "Spain"
df_us["COUNTRY"] = "US"

# Combine datasets
df = pd.concat([df_spain, df_us], axis=0).reset_index()

In [14]:
# See dataframe
df

,TIME_PERIOD,consumer_price,exchange_rate,industrial_production,international_reserves (M),COUNTRY
0,1959-M12,2.525752,60.0,NaN,200.141010,Spain
1,1960-M01,2.505928,60.0,NaN,233.260190,Spain
2,1960-M02,2.503895,60.0,NaN,253.215610,Spain
3,1960-M03,2.500846,60.0,NaN,299.282480,Spain
4,1960-M04,2.502879,60.0,NaN,326.282480,Spain
...,...,...,...,...,...,...
741,1990-M08,60.351608,NaN,NaN,169458.603357,US
742,1990-M09,60.856066,NaN,NaN,175533.733633,US
743,1990-M10,61.222946,NaN,NaN,170874.784032,US
744,1990-M11,61.360525,NaN,NaN,172797.695574,US


The IMF provides the nominal exchange rate as National Currency per US Dollar (period average)

This means Eₜ is defined as domestic currency per 1 USD, i.e. Spain currency / USD

Therefore, a higher Eₜ means a depreciation of Spain currency and an appreciation of USD

In [15]:
# Construct nominal exchange rate growth for Spain
df["nominal_exchange_rate_growth"] = (
    df.groupby("COUNTRY")["exchange_rate"]
      .transform(lambda x: np.log(x) - np.log(x.shift(1)))
)

There is no exchange rate data for US so `groupby` will not result error

### The monthly growth in the real exchange rate

As our nominal exchange rate is in the format of Spain currency / USD, so to calculate real exchange rate, we can use

ϵ = (E × Pᵤ) / Pₛ

where Pᵤ is the price level in US, and Pₛ is the price level in Spain

In [16]:
# Get US consumer price out for easy calculation of real exchange rate
us_consumer_price = (
    df[df["COUNTRY"] == "US"]
    [["TIME_PERIOD", "consumer_price"]]
    .rename(columns={"consumer_price": "consumer_price_us"})
)

# Merge dataframe
df = df.merge(us_consumer_price, on="TIME_PERIOD", how="left")

# Calculate real exchange rate for Spain
df["real_exchange_rate"] = (
    df["exchange_rate"] * df["consumer_price_us"] / df["consumer_price"]
)

# Calculate real exchange rate growth for Spain
df["real_exchange_rate_growth"] = (
    df.groupby("COUNTRY")["real_exchange_rate"]
      .transform(lambda x: np.log(x) - np.log(x.shift(1)))
)

# Drop real_exchange_rate and consumer_price_us that is not ask
df = df.drop(columns=["real_exchange_rate", "consumer_price_us"])

There is no exchange rate data for US so calculate real exchange rate and `groupby` will not result error

### An index of the real exchange rate

In [17]:
# Collect Spain data
spain = df[df["COUNTRY"] == "Spain"].copy()

# Calculate cumulative log level
spain["real_exchange_rate_log_level"] = (spain["real_exchange_rate_growth"]
                                     .fillna(0)
                                     .cumsum()
)

# Calculate cumulative level
spain["real_exchange_rate_level"] = np.exp(
    spain["real_exchange_rate_log_level"]
)

# Set index
spain["real_exchange_rate_index"] = (
    spain["real_exchange_rate_level"] /
    spain.loc[
        spain["TIME_PERIOD"] == "1990-M12", "real_exchange_rate_level"
    ].iloc[0]
)

# Merge
df = df.merge(
    spain[["COUNTRY", "TIME_PERIOD", "real_exchange_rate_index"]],
    on=["COUNTRY", "TIME_PERIOD"],
    how="left"
)

### The monthly inflation rate

In [18]:
# Construct monthly inflation rate for both Spain and the US
df["inflation"] = (
    df.groupby("COUNTRY")["consumer_price"]
      .transform(lambda x: np.log(x) - np.log(x.shift(1)))
)

### The monthly growth in industrial production

In [19]:
# Construct industrial production growth for Spain
df["industrial_production_growth_month"] = (
    df.groupby("COUNTRY")["industrial_production"]
      .transform(lambda x: np.log(x) - np.log(x.shift(1)))
)

### The growth in industrial production versus 12 months ago

In [20]:
# Construct yearly industrial production growth for Spain
df["industrial_production_growth_year"] = (
    df.groupby("COUNTRY")["industrial_production"]
      .transform(lambda x: np.log(x) - np.log(x.shift(12)))
)

### An index of the value of international reserves

In [21]:
# Collect Spain data
spain = df[df["COUNTRY"] == "Spain"].copy()

# Calculate index
spain["international_reserves_index"] = (
    spain["international_reserves (M)"] /
    spain.loc[
        spain["TIME_PERIOD"] == "1960-M01", "international_reserves (M)"
    ].iloc[0] * 100
)

In [22]:
# Collect US data
us = df[df["COUNTRY"] == "US"].copy()

# Calculate index
us["international_reserves_index"] = (
    us["international_reserves (M)"] /
    us.loc[
        us["TIME_PERIOD"] == "1960-M01", "international_reserves (M)"
    ].iloc[0] * 100
)


In [23]:
# Concat together
international_reserves_index = pd.concat(
    [spain[["TIME_PERIOD","COUNTRY", "international_reserves_index"]],
     us[["TIME_PERIOD","COUNTRY", "international_reserves_index"]]],
    axis=0
)

# Merge
df = df.merge(
    international_reserves_index,
    on=["COUNTRY", "TIME_PERIOD"],
    how="left"
)

In [24]:
# Format date
df["TIME_PERIOD"] = df["TIME_PERIOD"].apply(
    lambda x: datetime.strptime(x, "%Y-M%m")
)

In [25]:
# Show the full dataset
pd.set_option("display.max_rows", None)
df


,TIME_PERIOD,consumer_price,exchange_rate,industrial_production,international_reserves (M),COUNTRY,nominal_exchange_rate_growth,real_exchange_rate_growth,real_exchange_rate_index,inflation,industrial_production_growth_month,industrial_production_growth_year,international_reserves_index
0,1959-12-01,2.525752,60.000000,NaN,200.141010,Spain,NaN,NaN,2.561868,NaN,NaN,NaN,85.801615
1,1960-01-01,2.505928,60.000000,NaN,233.260190,Spain,0.000000,0.004473,2.573352,-0.007880,NaN,NaN,100.000000
2,1960-02-01,2.503895,60.000000,NaN,253.215610,Spain,0.000000,0.004219,2.584231,-0.000812,NaN,NaN,108.555005
3,1960-03-01,2.500846,60.000000,NaN,299.282480,Spain,0.000000,0.001218,2.587382,-0.001218,NaN,NaN,128.304140
4,1960-04-01,2.502879,60.000000,NaN,326.282480,Spain,0.000000,0.002583,2.594074,0.000813,NaN,NaN,139.879197
5,1960-05-01,2.495255,60.000000,NaN,360.215610,Spain,0.000000,0.003051,2.602000,-0.003051,NaN,NaN,154.426527
6,1960-06-01,2.493221,60.000000,NaN,406.215610,Spain,0.000000,0.004200,2.612950,-0.000815,NaN,NaN,174.146994
7,1960-07-01,2.492204,60.000000,NaN,427.304770,Spain,0.000000,0.000408,2.614016,-0.000408,NaN,NaN,183.188040
8,1960-08-01,2.497288,60.000000,NaN,455.392240,Spain,0.000000,-0.002038,2.608695,0.002038,NaN,NaN,195.229302
9,1960-09-01,2.512536,60.000000,NaN,479.644000,Spain,0.000000,-0.006087,2.592863,0.006087,NaN,NaN,205.626172


In [26]:
# Reset
pd.reset_option("display.max_rows")

## Analysis